# Choi2018-JFM-SumBSM

### Title:
__Sum of all Black-Scholes-Merton models: An efficient pricing method for spread, basket, and Asian options__

### Author:
* Jaehyuk Choi ([@jaehyukchoi](https://github.com/jaehyukchoi))

### Abstract:
This paper proposes a unified framework for pricing spread, basket, and Asian options under the Black–Scholes–Merton model. The option price is expressed as a sum of single-factor BSM prices, whose first Brownian factor covers the dominant market risk, with the remaining factors integrated out via Gauss–Hermite quadrature. A control variate based on the forward price condition eliminates systematic quadrature error, yielding highly accurate "fast prices" with a small number of quadrature nodes. Numerical examples demonstrate errors on the order of 1e-4 to 1e-8 with moderate quadrature settings.

### Journal / DOI:
Journal of Futures Markets, 38(5), 627–644. https://doi.org/10.1002/fut.21837

### Keywords:
basket options, spread options, Asian options, Black–Scholes–Merton, Gauss–Hermite quadrature, control variate

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
### if you want to run on your modified PyFeng code.
#%load_ext autoreload
#%autoreload 2

In [ ]:
# Install pyfeng if not available (e.g., in Google Colab)
try:
    ### Uncomment below if you want to run on your modified code
    #import sys
    #sys.path.insert(sys.path.index('')+1, 'YOUR_LOCAL_PyFeng_PATH')
    import pyfeng as pf
except ImportError:
    # Install pyfeng if not available (e.g., in Google Colab)
    import subprocess
    subprocess.run(["pip", "install", "pyfeng"])
    import pyfeng as pf

## Notes
* `CP` = converged price (large quadrature order); `FP` = fast price with paper-suggested settings
* `FP Err` = FP − CP
* `BsmBasketChoi2018` implements Eq. (9)/(14)/(17): sum-of-BSM formula with delta/forward control variate
* `init_lowerbound` sets `n_quad=0` (single-factor lower bound, Eq. 9 only)
* `set_num_params(n_quad=[M])` fixes the number of Gauss–Hermite nodes per dimension
* `set_num_params(lam=λ)` uses Eq. (32) to auto-compute node counts from the scale-length ratio

---
## Table 4: Spread Option Set S1 — Varying Strike K
N=2, T=1, S=(100, 96), w=(+1, −1), σ=(20%, 10%), ρ=50%, q=5%, r=10%.  
Fast price uses M₂=4; errors ≈ 1e-8 (with control variate).

In [ ]:
sigma_S1 = np.array([0.20, 0.10])
spot_S1  = np.array([100.0, 96.0])
w_S1     = np.array([1.0, -1.0])
texp_S1, intr_S1, divr_S1 = 1.0, 0.10, 0.05

strike_S1 = np.arange(0.0, 4.1, 0.4)

# CP from paper (Table 4)
cp4 = np.array([8.5132252, 8.3124607, 8.1149938, 7.9208198, 7.7299325,
                7.5423239, 7.3579843, 7.1769024, 6.9990651, 6.8244581, 6.6530651])

m_fp = pf.BsmBasketChoi2018(sigma_S1, rho=0.50, weight=w_S1, intr=intr_S1, divr=divr_S1)
m_fp.set_num_params(n_quad=[4])   # fast: M₂=4
fp4 = m_fp.price(strike_S1, spot_S1, texp_S1)

df4 = pd.DataFrame({'K': strike_S1, 'CP': cp4, 'FP (M₂=4)': fp4, 'FP Err': fp4 - cp4})
df4.style.format({'K': '{:.1f}', 'CP': '{:.7f}', 'FP (M₂=4)': '{:.7f}', 'FP Err': '{:.2e}'})

---
## Table 5: Spread Option Set S2 — Varying Correlation ρ
N=2, T=1, S=(200, 100), w=(+1, −1), K=100, σ=(15%, 30%), q=r=0.  
Fast price uses λ=9; errors ≈ 5e-8.

In [ ]:
sigma_S2 = np.array([0.15, 0.30])
spot_S2  = np.array([200.0, 100.0])
w_S2     = np.array([1.0, -1.0])
texp_S2, strike_S2 = 1.0, 100.0

rhos_S2 = [0.90, 0.70, 0.50, 0.30, 0.10, -0.10, -0.30, -0.50, -0.70, -0.90]

# CP from paper (Table 5)
cp5 = np.array([5.4792720, 9.3209439, 11.9804918, 14.1425869, 16.0102190,
                17.6770249, 19.1954201, 20.5982705, 21.9077989, 23.1398674])

fp5 = []
for rho in rhos_S2:
    m = pf.BsmBasketChoi2018(sigma_S2, rho=rho, weight=w_S2)
    m.set_num_params(lam=9)
    fp5.append(m.price(strike_S2, spot_S2, texp_S2))
fp5 = np.array(fp5)

df5 = pd.DataFrame({'ρ': rhos_S2, 'CP': cp5, 'FP (λ=9)': fp5, 'FP Err': fp5 - cp5})
df5.style.format({'ρ': '{:.2f}', 'CP': '{:.7f}', 'FP (λ=9)': '{:.7f}', 'FP Err': '{:.2e}'})

---
## Table 6: Basket Option Set B1 — Varying Strike K
N=4, T=5, S=100 (equal), w=1/4, σ=40%, ρ=50%, r=q=0.  
Fast price uses λ=9 (M=125 nodes); errors ≤ 1.5e-4.

In [ ]:
sigma_B1 = 0.4 * np.ones(4)
spot_B1  = 100.0 * np.ones(4)
texp_B1  = 5.0

strike_B1 = np.arange(50.0, 151.0, 10.0)

# CP from paper (Table 6)
cp6 = np.array([54.3101761, 47.4811265, 41.5225192, 36.3517843, 31.8768032,
                28.0073695, 24.6605295, 21.7625789, 19.2493294, 17.0655420, 15.1640103])

m_fp = pf.BsmBasketChoi2018(sigma_B1, rho=0.5)
m_fp.set_num_params(lam=9)
fp6 = m_fp.price(strike_B1, spot_B1, texp_B1)

df6 = pd.DataFrame({'K': strike_B1.astype(int), 'CP': cp6, 'FP (λ=9)': fp6, 'FP Err': fp6 - cp6})
df6.style.format({'CP': '{:.7f}', 'FP (λ=9)': '{:.7f}', 'FP Err': '{:.2e}'})

---
## Table 7: Basket Option Set B1 — Varying Correlation ρ
N=4, T=5, K=100, S=100, w=1/4, σ=40%, r=q=0.  
Fast price uses λ=9; errors ≤ 2e-4 for ρ ≤ 0.80, ~3e-3 for ρ=0.95 (only M=8 nodes).

In [ ]:
rhos_B1 = [-0.10, 0.10, 0.30, 0.50, 0.70, 0.80, 0.95]
strike_B1atm = 100.0

# CP from paper (Table 7)
cp7 = np.array([17.7569163, 21.6920965, 25.0292992, 28.0073695, 30.7427415, 32.0412265, 33.9186874])

fp7 = []
for rho in rhos_B1:
    m = pf.BsmBasketChoi2018(sigma_B1, rho=rho)
    m.set_num_params(lam=9)
    fp7.append(m.price(strike_B1atm, spot_B1, texp_B1))
fp7 = np.array(fp7)

df7 = pd.DataFrame({'ρ': rhos_B1, 'CP': cp7, 'FP (λ=9)': fp7, 'FP Err': fp7 - cp7})
df7.style.format({'ρ': '{:.2f}', 'CP': '{:.7f}', 'FP (λ=9)': '{:.7f}', 'FP Err': '{:.2e}'})

---
## Table 8: Basket Option Set B1 — Inhomogeneous Volatilities
N=4, T=5, K=100, S=100, w=1/4, ρ=50%, r=q=0; σ₁=40% fixed, σ₂=σ₃=σ₄ vary.  
**Note**: exact Table 8 parameterization TBD; CP computed with λ=13 (not from paper).  
Fast price uses λ=9.

In [ ]:
# Inhomogeneous σ: asset 1 fixed at 40%, assets 2-4 vary together
sigma2_vals = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]

# CP from high-accuracy run (Table 8)
cp8 = np.array([12.6242390, 17.6093739, 22.8093657, 28.0073712,
                33.1085375, 38.0460432, 42.7634761, 47.2095472])

fp8, sigma_labels = [], []
for s2 in sigma2_vals:
    sig = np.array([0.40, s2, s2, s2])
    m = pf.BsmBasketChoi2018(sig, rho=0.5)
    m.set_num_params(lam=9)
    fp8.append(m.price(100.0, spot_B1, texp_B1))
    sigma_labels.append(f'(40,{int(s2*100)},{int(s2*100)},{int(s2*100)})%')
fp8 = np.array(fp8)

df8 = pd.DataFrame({'σ (%)': sigma_labels, 'CP': cp8, 'FP (λ=9)': fp8, 'FP Err': fp8 - cp8})
df8.style.format({'CP': '{:.7f}', 'FP (λ=9)': '{:.7f}', 'FP Err': '{:.2e}'})

---
## Table 9: Basket Option Set B2 — 7-Asset Basket, Varying K and T
N=7, w and σ inhomogeneous, full correlation matrix, r=6.3%, q inhomogeneous.  
Fast price uses λ=3; errors ≤ 1e-4.

In [ ]:
# B2 parameters (Table 3 of paper)
sigma_B2  = np.array([11.55, 20.68, 14.53, 17.99, 15.59, 14.62, 15.68]) / 100
weight_B2 = np.array([0.10,  0.15,  0.15,  0.05,  0.20,  0.10,  0.25])
divr_B2   = np.array([1.69,  2.39,  1.36,  1.92,  0.81,  3.62,  1.66]) / 100
intr_B2   = 0.063
cor_m_B2  = np.array([
    [ 1.00,  0.35,  0.10,  0.27,  0.04,  0.17,  0.71],
    [ 0.35,  1.00,  0.39,  0.27,  0.50, -0.08,  0.15],
    [ 0.10,  0.39,  1.00,  0.53,  0.70, -0.23,  0.09],
    [ 0.27,  0.27,  0.53,  1.00,  0.46, -0.22,  0.32],
    [ 0.04,  0.50,  0.70,  0.46,  1.00, -0.29,  0.13],
    [ 0.17, -0.08, -0.23, -0.22, -0.29,  1.00, -0.03],
    [ 0.71,  0.15,  0.09,  0.32,  0.13, -0.03,  1.00],
])
spot_B2 = 100.0 * np.ones(7)

strikes_B2 = [80.0, 100.0, 120.0]
texps_B2   = [0.5, 1.0, 2.0, 3.0]

In [ ]:
# CP computed with lam=7 (high-accuracy reference; rows = K [80,100,120], cols = T [0.5,1,2,3])
cp9 = np.array([
    [21.6022546, 23.1411627, 26.0424328, 28.6992602],  # K=80
    [ 3.8828353,  6.2216810, 10.2156012, 13.7425580],  # K=100
    [ 0.0235189,  0.3535584,  2.0570044,  4.4578389],  # K=120
])

fp9 = np.zeros_like(cp9)
for j, texp in enumerate(texps_B2):
    m = pf.BsmBasketChoi2018(
        sigma_B2, cor_m=cor_m_B2, weight=weight_B2, intr=intr_B2, divr=divr_B2
    )
    m.set_num_params(lam=3)
    for i, K in enumerate(strikes_B2):
        fp9[i, j] = m.price(K, spot_B2, texp)

col_cp  = pd.MultiIndex.from_product([['CP'],       texps_B2], names=['', 'T'])
col_fp  = pd.MultiIndex.from_product([['FP (λ=3)'], texps_B2], names=['', 'T'])
col_err = pd.MultiIndex.from_product([['FP Err'],   texps_B2], names=['', 'T'])

df9 = pd.DataFrame(
    np.hstack([cp9, fp9, fp9 - cp9]),
    index=pd.Index(strikes_B2, name='K'),
    columns=col_cp.append(col_fp).append(col_err)
)
df9.style.format(
    {c: '{:.7f}' for c in df9.columns if c[0] in ('CP', 'FP (λ=3)')}
).format(
    {c: '{:.2e}' for c in df9.columns if c[0] == 'FP Err'}
)